# CAMEL Playground notebook (SENG 691 Group 3, Phase 2)

Phase 1 built the Playground: two CAMEL agents role-play a task while a person
watches, guides, and instructs them. Phase 2 made that Playground continuous,
supervisable, and reliable, with three enhancements:

1. **Persistent session history.** Sessions are saved to a local SQLite
   database as they run, so a session can be reopened after a restart with its
   task, roles, transcript, guidance, and settings intact.
2. **Progress monitoring and run controls.** A deterministic local monitor
   flags repetition, stalls, drift from the task, and possible completion, and
   the run controls add a configurable turn limit and pause and resume.
3. **Model configuration and error recovery.** A preflight check validates the
   provider, model, and credentials before a run starts, and failed calls are
   classified and retried with backoff when the failure is temporary.

This notebook is self-contained. The three Phase 2 modules are standard
library only, so their full source is included below in its own cells, exactly
as it ships in the repository's `demo` folder, and the demonstrations that
follow run against that code. Any machine with Python 3.10 or later can run
the whole notebook top to bottom: no installs, no API key, no repository
clone needed. The one thing it does not embed is the Playground app itself,
which needs the CAMEL framework and Gradio; where the app wraps these modules
(a live session in the browser, the turn limit, pause and resume), the
repository's `demo/test_phase2.py` exercises that layer, and the closing cell
says which parts are shown here directly.

## The Phase 2 modules, inlined

The next three cells are the team's modules, verbatim. Nothing in them needs
to be edited; running them defines the session store, the progress monitor,
and the model configuration helpers that the rest of the notebook uses.

### `session_store.py`: the SQLite session store (Enhancement 1)

In [1]:
"""Persistent session history for the CAMEL Playground (SENG 691 Phase 2, Enhancement 1).

Phase 1 kept a session only in Gradio memory: restarting the app lost the task,
the roles, the transcript, and the model settings. This module stores them in a
small SQLite database (Python standard library only, no new dependency), using
the schema the team proposed in the Phase 1 ER diagram: a session row (the
AGENT_SESSION entity, with the task and model configuration fields it points
to), one row per transcript message (MESSAGE), and one row per human guidance
note (GUIDANCE).

The Playground writes through to this store as a session runs, so what is on
disk is always the current session, and any saved session can be reopened after
a restart with its task, roles, transcript, guidance, model configuration, turn
count, and completion status intact.
"""

from __future__ import annotations

import sqlite3
from datetime import datetime
from pathlib import Path

SCHEMA = """
CREATE TABLE IF NOT EXISTS sessions (
    session_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at     TEXT NOT NULL,
    updated_at     TEXT NOT NULL,
    task_original  TEXT NOT NULL,
    specified_task TEXT NOT NULL DEFAULT '',
    assistant_role TEXT NOT NULL,
    user_role      TEXT NOT NULL,
    platform       TEXT NOT NULL DEFAULT '',
    model_name     TEXT NOT NULL DEFAULT '',
    mode           TEXT NOT NULL DEFAULT '',
    concise        INTEGER NOT NULL DEFAULT 1,
    turn           INTEGER NOT NULL DEFAULT 0,
    status         TEXT NOT NULL DEFAULT 'in_progress'
);
CREATE TABLE IF NOT EXISTS messages (
    session_id INTEGER NOT NULL REFERENCES sessions(session_id) ON DELETE CASCADE,
    seq        INTEGER NOT NULL,
    speaker    TEXT NOT NULL,
    text       TEXT NOT NULL,
    PRIMARY KEY (session_id, seq)
);
CREATE TABLE IF NOT EXISTS guidance (
    session_id INTEGER NOT NULL REFERENCES sessions(session_id) ON DELETE CASCADE,
    seq        INTEGER NOT NULL,
    turn       INTEGER NOT NULL,
    text       TEXT NOT NULL,
    PRIMARY KEY (session_id, seq)
);
"""

# Session fields a reopen is expected to restore. The Phase 2 evidence metric
# for this enhancement counts how many of these come back exactly.
RESTORED_FIELDS = (
    "task_original", "specified_task", "assistant_role", "user_role",
    "platform", "model_name", "mode", "concise", "turn", "status",
)


def _now() -> str:
    return datetime.now().isoformat(timespec="seconds")


class SessionStore:
    """A tiny write-through store for Playground sessions."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.path)
        self.db.row_factory = sqlite3.Row
        self.db.execute("PRAGMA foreign_keys = ON")
        self.db.executescript(SCHEMA)
        self.db.commit()

    def close(self) -> None:
        self.db.close()

    # ----- writing -----------------------------------------------------------
    def create_session(self, *, task_original: str, specified_task: str,
                       assistant_role: str, user_role: str, platform: str,
                       model_name: str, mode: str, concise: bool) -> int:
        now = _now()
        cur = self.db.execute(
            "INSERT INTO sessions (created_at, updated_at, task_original,"
            " specified_task, assistant_role, user_role, platform, model_name,"
            " mode, concise) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (now, now, task_original, specified_task, assistant_role, user_role,
             platform, model_name, mode, 1 if concise else 0),
        )
        self.db.commit()
        return int(cur.lastrowid)

    def append_message(self, session_id: int, speaker: str, text: str) -> None:
        seq = self.db.execute(
            "SELECT COALESCE(MAX(seq), -1) + 1 FROM messages WHERE session_id = ?",
            (session_id,),
        ).fetchone()[0]
        self.db.execute(
            "INSERT INTO messages (session_id, seq, speaker, text) VALUES (?, ?, ?, ?)",
            (session_id, seq, speaker, text),
        )
        self._touch(session_id)

    def add_guidance(self, session_id: int, turn: int, text: str) -> None:
        seq = self.db.execute(
            "SELECT COALESCE(MAX(seq), -1) + 1 FROM guidance WHERE session_id = ?",
            (session_id,),
        ).fetchone()[0]
        self.db.execute(
            "INSERT INTO guidance (session_id, seq, turn, text) VALUES (?, ?, ?, ?)",
            (session_id, seq, turn, text),
        )
        self._touch(session_id)

    def update_state(self, session_id: int, *, turn: int | None = None,
                     status: str | None = None) -> None:
        sets, values = [], []
        if turn is not None:
            sets.append("turn = ?")
            values.append(turn)
        if status is not None:
            sets.append("status = ?")
            values.append(status)
        if not sets:
            return
        values.append(session_id)
        self.db.execute(
            f"UPDATE sessions SET {', '.join(sets)} WHERE session_id = ?", values
        )
        self._touch(session_id)

    def _touch(self, session_id: int) -> None:
        self.db.execute(
            "UPDATE sessions SET updated_at = ? WHERE session_id = ?",
            (_now(), session_id),
        )
        self.db.commit()

    # ----- reading -------------------------------------------------------------
    def get_session(self, session_id: int) -> dict | None:
        row = self.db.execute(
            "SELECT * FROM sessions WHERE session_id = ?", (session_id,)
        ).fetchone()
        if row is None:
            return None
        data = dict(row)
        data["concise"] = bool(data["concise"])
        data["messages"] = [
            (r["speaker"], r["text"])
            for r in self.db.execute(
                "SELECT speaker, text FROM messages WHERE session_id = ? ORDER BY seq",
                (session_id,),
            )
        ]
        data["guidance"] = [
            dict(r)
            for r in self.db.execute(
                "SELECT seq, turn, text FROM guidance WHERE session_id = ? ORDER BY seq",
                (session_id,),
            )
        ]
        return data

    def list_sessions(self, limit: int = 50) -> list[dict]:
        rows = self.db.execute(
            "SELECT session_id, created_at, updated_at, task_original,"
            " assistant_role, user_role, turn, status, mode FROM sessions"
            " ORDER BY updated_at DESC LIMIT ?",
            (limit,),
        ).fetchall()
        return [dict(r) for r in rows]

    def delete_session(self, session_id: int) -> None:
        self.db.execute("DELETE FROM messages WHERE session_id = ?", (session_id,))
        self.db.execute("DELETE FROM guidance WHERE session_id = ?", (session_id,))
        self.db.execute("DELETE FROM sessions WHERE session_id = ?", (session_id,))
        self.db.commit()

### `progress_monitor.py`: the deterministic progress monitor (Enhancement 2)

In [2]:
"""Progress and termination signals for the CAMEL Playground (Phase 2, Enhancement 2).

Phase 1's Narrator was a language model asked, in prose, whether the agents
were repeating themselves, drifting, or stalling. That costs an extra model
call, and its judgement cannot be tested. This module computes the same signals
locally and deterministically from the transcript, with no model call:

  * repetition: the newest message from an agent is nearly identical to one of
    that agent's recent messages (token-set Jaccard similarity).
  * stall: several agent turns in a row add almost no new words to the
    conversation (novelty below a floor), so the run is going in circles.
  * drift: the recent exchanges barely mention the task's own keywords.
  * possible completion: the instructing side uses explicit completion
    language ("task complete", "all done", ...) without the formal done token.

The Playground shows each flag in the chat as a Narrator progress check and
counts them in the status strip. The configurable turn limit and the pause and
resume controls live in playground.py; the detection lives here so it can be
unit tested on scripted transcripts.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field

STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "with",
    "is", "are", "be", "we", "you", "i", "it", "this", "that", "as", "at",
    "by", "from", "our", "your", "their", "will", "would", "can", "could",
    "should", "about", "into", "over", "after", "before", "between",
}

COMPLETION_PHRASES = (
    "task complete", "task is complete", "all done", "we are done",
    "that completes", "successfully completed", "goal achieved",
)


def tokens(text: str) -> set[str]:
    """Lowercased word tokens with markdown and punctuation stripped."""
    return set(re.findall(r"[a-z0-9']+", (text or "").lower()))


def keywords(text: str) -> set[str]:
    """The content words of a text: its tokens minus stopwords and short words."""
    return {t for t in tokens(text) if t not in STOPWORDS and len(t) > 2}


def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


@dataclass
class ProgressFlag:
    kind: str        # repetition | stall | drift | possible_completion
    turn: int
    detail: str

    def line(self) -> str:
        return f"Progress check: {self.detail}"


@dataclass
class ProgressMonitor:
    """Watches a transcript grow and flags repetition, stalls, and drift."""

    task_text: str = ""
    repeat_threshold: float = 0.8     # Jaccard at or above this = a repeat
    novelty_floor: float = 0.15       # below this share of new words = no progress
    stall_window: int = 2             # consecutive low-novelty turns = a stall
    drift_floor: float = 0.05         # task-keyword share below this = drift

    _history: dict = field(default_factory=dict)   # speaker -> list of token sets
    _seen: set = field(default_factory=set)        # every token seen so far
    _low_novelty_streak: int = 0
    _stall_reported: bool = False
    _last_drift_turn: int = -10
    flags: list = field(default_factory=list)

    def __post_init__(self) -> None:
        self._task_keywords = keywords(self.task_text)

    # ----- one call per completed turn --------------------------------------
    def observe(self, turn: int, entries: list, assistant_role: str,
                user_role: str) -> list:
        """Check the newest turn's agent messages; return any new flags."""
        new_flags: list[ProgressFlag] = []
        agent_entries = [
            (who, text) for who, text in entries if who in (assistant_role, user_role)
        ]
        for who, text in agent_entries:
            toks = tokens(text)
            prior = self._history.setdefault(who, [])
            for old in prior[-3:]:
                sim = jaccard(toks, old)
                if sim >= self.repeat_threshold:
                    new_flags.append(ProgressFlag(
                        "repetition", turn,
                        f"{who} is repeating an earlier message "
                        f"({sim:.0%} similar) at turn {turn}."))
                    break
            prior.append(toks)
            if who == user_role:
                low = text.lower()
                if any(p in low for p in COMPLETION_PHRASES):
                    new_flags.append(ProgressFlag(
                        "possible_completion", turn,
                        f"{user_role} used completion language at turn {turn} "
                        "without the formal done signal."))
        if agent_entries:
            new_flags += self._novelty_check(turn, agent_entries)
            new_flags += self._drift_check(turn, agent_entries)
        self.flags.extend(new_flags)
        return new_flags

    def _novelty_check(self, turn: int, agent_entries: list) -> list:
        combined: set = set()
        for _, text in agent_entries:
            combined |= tokens(text)
        if not combined:
            return []
        novelty = len(combined - self._seen) / len(combined)
        self._seen |= combined
        if novelty < self.novelty_floor:
            self._low_novelty_streak += 1
        else:
            self._low_novelty_streak = 0
            self._stall_reported = False
        if self._low_novelty_streak >= self.stall_window and not self._stall_reported:
            self._stall_reported = True
            return [ProgressFlag(
                "stall", turn,
                f"The last {self._low_novelty_streak} turns added almost no new "
                f"content. The run looks stalled at turn {turn}.")]
        return []

    def _drift_check(self, turn: int, agent_entries: list) -> list:
        if turn < 3 or not self._task_keywords:
            return []
        if turn - self._last_drift_turn < 2:
            return []
        recent: set = set()
        for _, text in agent_entries:
            recent |= tokens(text)
        share = len(recent & self._task_keywords) / len(self._task_keywords)
        if share < self.drift_floor:
            self._last_drift_turn = turn
            return [ProgressFlag(
                "drift", turn,
                f"Recent exchanges barely mention the task itself "
                f"({share:.0%} of its key words) at turn {turn}. "
                "The agents may be drifting from the goal.")]
        return []

    # ----- reporting -----------------------------------------------------------
    def counts(self) -> dict:
        out: dict = {}
        for flag in self.flags:
            out[flag.kind] = out.get(flag.kind, 0) + 1
        return out

    def summary_line(self) -> str:
        counts = self.counts()
        if not counts:
            return "Progress flags: none so far."
        parts = ", ".join(f"{kind} x{n}" for kind, n in sorted(counts.items()))
        return f"Progress flags: {parts}. Latest: {self.flags[-1].detail}"

### `model_config.py`: preflight checks, error classification, retry (Enhancement 3)

In [3]:
"""Centralized model configuration, preflight checks, and error recovery
(SENG 691 Phase 2, Enhancement 3).

Phase 1 resolved the provider, model, and API key in two different places
(`make_model` and `choose_model`), and a configuration mistake only surfaced
after the user pressed Start and the first model call failed mid-run. This
module puts the checks in one place:

  * `preflight` validates the resolved configuration before a run starts:
    the provider is known, a model name resolves, the API key is present,
    and (optionally) the model object can be constructed. Each check reports
    pass or fail with the fix, so a bad configuration is caught on the setup
    screen instead of mid-run.
  * `classify_error` and `guidance_for` turn provider errors into a category
    (rate limit, rejected key, unknown model, timeout, connection) and one
    line of advice, the same wording the Playground has always shown.
  * `call_with_retry` retries the transient categories (rate limit, timeout,
    connection) with exponential backoff and reports each retry through a
    callback, while permanent errors (a rejected key, an unknown model) fail
    at once with their guidance instead of being retried pointlessly.
"""

from __future__ import annotations

import os
import time
from dataclasses import dataclass, field

TRANSIENT = ("rate_limit", "timeout", "connection")


def classify_error(exc: Exception) -> str:
    """Bucket a provider error: rate_limit, auth, model_not_found, timeout,
    connection, or unknown. Mirrors the cases playground.friendly() handles."""
    low = str(exc).lower()
    if any(s in low for s in ("429", "rate limit", "rate_limit", "quota",
                              "too many requests")):
        return "rate_limit"
    if any(s in low for s in ("401", "invalid api key", "incorrect api key",
                              "unauthorized", "403", "forbidden")):
        return "auth"
    if any(s in low for s in ("404", "model not found", "does not exist",
                              "no endpoints")):
        return "model_not_found"
    if "timed out" in low or "timeout" in low:
        return "timeout"
    if "connection" in low or "connect" in low:
        return "connection"
    return "unknown"


def guidance_for(category: str) -> str:
    """One line of advice per error category, for a non-developer reader."""
    return {
        "rate_limit": ("The model provider's free-tier limit was reached. Wait a "
                       "minute and try again, or switch provider under Advanced "
                       "(free limits: OpenRouter 50 requests/day until $10 of "
                       "credit is bought, then 1,000/day; Groq about 1,000/day)."),
        "auth": "The API key was rejected. Check the key for the chosen provider.",
        "model_not_found": ("That model name is not available on this provider; "
                            "clear the Model box to use the provider default."),
        "timeout": "The provider did not answer in time. Try again.",
        "connection": "The provider could not be reached. Check the connection and try again.",
        "unknown": "Something went wrong calling the model. See the details below.",
    }.get(category, "Something went wrong calling the model.")


@dataclass
class Check:
    name: str
    ok: bool
    detail: str


@dataclass
class PreflightResult:
    checks: list = field(default_factory=list)

    @property
    def ok(self) -> bool:
        return all(c.ok for c in self.checks)

    def render(self) -> str:
        lines = ["**Model setup check**"]
        for c in self.checks:
            mark = "OK" if c.ok else "Needs attention"
            lines.append(f"- {c.name}: {mark}. {c.detail}")
        lines.append("**Ready to start.**" if self.ok
                     else "**Fix the items above before starting.**")
        return "\n".join(lines)


def preflight(platforms: dict, platform: str | None, model_name: str | None,
              stub: bool = False, env: dict | None = None, create=None) -> PreflightResult:
    """Validate a resolved model configuration before a run starts.

    `platforms` is the shared table {name: (key_env_var, default_model)}.
    `create`, when given, is called as create(platform, model_name) to prove
    the model object can be constructed; it spends no tokens.
    """
    env = os.environ if env is None else env
    result = PreflightResult()
    if stub:
        result.checks.append(Check(
            "Offline stub", True,
            "Offline test mode is on: canned replies, no key or provider needed."))
        return result

    platform = (platform or env.get("MODEL_PLATFORM") or "openai").lower()
    known = platform in platforms
    key_var, default_model = platforms.get(platform, (f"{platform.upper()}_API_KEY", None))
    resolved_model = model_name or env.get("MODEL_NAME") or default_model

    result.checks.append(Check(
        "Provider", known or bool(resolved_model),
        f"'{platform}' is a configured provider." if known
        else f"'{platform}' is not in the provider table; it will be used as a "
             "custom platform with the model name given."))
    result.checks.append(Check(
        "Model name", bool(resolved_model),
        f"Model resolves to '{resolved_model}'." if resolved_model
        else "No model name given and this provider has no default. "
             "Enter a model name."))
    if key_var is None:
        result.checks.append(Check(
            "API key", True, "This provider needs no API key (local model)."))
    else:
        present = bool((env.get(key_var) or "").strip())
        result.checks.append(Check(
            "API key", present,
            f"{key_var} is set." if present
            else f"{key_var} is not set. Put it in .env, or tick 'offline stub' "
                 "to try the UI without a key."))
    if create is not None and result.ok:
        try:
            create(platform, resolved_model)
            result.checks.append(Check(
                "Model object", True,
                "The model object was constructed. The key itself is verified "
                "by the provider on the first call."))
        except Exception as exc:  # construction failed: report, do not raise
            result.checks.append(Check(
                "Model object", False,
                f"{guidance_for(classify_error(exc))} Details: {str(exc)[:200]}"))
    return result


def call_with_retry(fn, attempts: int = 3, base_delay: float = 0.5,
                    max_delay: float = 8.0, sleep=time.sleep, on_retry=None):
    """Call fn(), retrying transient provider errors with exponential backoff.

    `on_retry(attempt, category, exc)` is called before each retry so the
    caller can tell the user what happened. Permanent errors raise at once.
    """
    last_exc: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            category = classify_error(exc)
            if category not in TRANSIENT or attempt == attempts:
                raise
            if on_retry is not None:
                on_retry(attempt, category, exc)
            sleep(min(base_delay * 2 ** (attempt - 1), max_delay))
    raise last_exc  # pragma: no cover - the loop always returns or raises

## Enhancement 1: persistent session history

In Phase 1 a session lived only in the app's memory, so a restart lost the
conversation. The session store now writes every session to SQLite as it runs,
which is the component the next cell exercises: it creates a session, records
two turns of a conversation, a guidance note, and an instruction, then closes
the store and reopens the same database file fresh, the way the app does after
a restart, and compares what came back, field by field.

In [4]:
import tempfile
import time
from pathlib import Path

db_path = str(Path(tempfile.mkdtemp()) / "sessions.db")

store = SessionStore(db_path)
sid = store.create_session(
    task_original="Write a small trading bot in Python",
    specified_task="Write a small trading bot in Python with risk checks",
    assistant_role="Python Programmer", user_role="Stock Trader",
    platform="stub", model_name="stub", mode="stub", concise=True)
store.append_message(sid, "Stock Trader", "Write a small trading bot in Python")
store.append_message(sid, "Python Programmer", "I will start with the data feed and a moving average signal.")
store.append_message(sid, "Stock Trader", "Add position sizing next")
store.append_message(sid, "Python Programmer", "Position sizing added, capped at two percent per trade.")
store.add_guidance(sid, 2, "Focus on risk checks next")
store.append_message(sid, "Stock Trader", "Add a function that computes a moving average")
store.update_state(sid, turn=3, status="in_progress")
before = store.get_session(sid)
store.close()
print(f"Session {sid} saved: {len(before['messages'])} messages, turn {before['turn']}")

t0 = time.perf_counter()
store2 = SessionStore(db_path)
after = store2.get_session(sid)
restore_seconds = time.perf_counter() - t0

matched = sum(1 for f in RESTORED_FIELDS if after[f] == before[f])
print(f"Fields restored exactly: {matched}/{len(RESTORED_FIELDS)} "
      f"(Phase 1 baseline: 0/{len(RESTORED_FIELDS)})")
print(f"Transcript restored exactly: {after['messages'] == before['messages']} "
      f"({len(after['messages'])} messages), restore took {restore_seconds:.3f} s")
print(f"Guidance restored: {after['guidance'][0]['text']!r}")

store2.append_message(sid, "Python Programmer", "Moving average function added with a risk check.")
store2.update_state(sid, turn=4)
continued = store2.get_session(sid)
print(f"The reopened session continues: turn {before['turn']} -> {continued['turn']}, "
      f"{len(continued['messages'])} messages")
store2.close()

Session 1 saved: 5 messages, turn 3
Fields restored exactly: 10/10 (Phase 1 baseline: 0/10)
Transcript restored exactly: True (5 messages), restore took 0.001 s
Guidance restored: 'Focus on risk checks next'
The reopened session continues: turn 3 -> 4, 6 messages


## Enhancement 2: progress monitoring

The monitor reads the transcript after every turn and applies simple word
overlap rules. It makes no model call, so it costs nothing and its decisions
can be checked by hand. The next cell feeds it a scripted conversation in
which the assistant repeats the same haiku, and prints the flags as the
Narrator lines the app shows. The run controls that sit beside the monitor in
the app, the turn limit and pause and resume, are exercised against the live
Playground in the repository's test suite.

In [5]:
A, U = "Poet", "Editor"
haiku = "Autumn leaves drift down softly over the quiet garden path"
mon = ProgressMonitor(task_text="Write a haiku about autumn leaves falling")

mon.observe(1, [(U, "Write a haiku about autumn leaves"), (A, haiku)], A, U)
for turn in (2, 3, 4):
    flags = mon.observe(turn, [(U, "Try another version please"), (A, haiku)], A, U)
    for f in flags:
        print(f.line())
print(mon.summary_line())

drift = ProgressMonitor(task_text="Plan a weekend trip to Montreal on a student budget")
drift.observe(1, [(U, "Plan the Montreal weekend trip"),
                  (A, "We can budget the train and a hostel in Montreal")], A, U)
drift.observe(2, [(U, "What about food costs for the weekend"),
                  (A, "Street food keeps the Montreal budget low")], A, U)
for f in drift.observe(3, [(U, "Explain quantum entanglement instead"),
                           (A, "Quantum particles share correlated states instantly")], A, U):
    print(f.line())

Progress check: Poet is repeating an earlier message (100% similar) at turn 2.
Progress check: Editor is repeating an earlier message (100% similar) at turn 3.
Progress check: Poet is repeating an earlier message (100% similar) at turn 3.
Progress check: Editor is repeating an earlier message (100% similar) at turn 4.
Progress check: Poet is repeating an earlier message (100% similar) at turn 4.
Progress check: The last 2 turns added almost no new content. The run looks stalled at turn 4.
Progress flags: repetition x5, stall x1. Latest: The last 2 turns added almost no new content. The run looks stalled at turn 4.
Progress check: Recent exchanges barely mention the task itself (0% of its key words) at turn 3. The agents may be drifting from the goal.


## Enhancement 3: model configuration and error recovery

In Phase 1 a setup mistake, such as a missing key or a mistyped model name,
only surfaced after Start, when the first model call failed mid-run. The
preflight check now validates the setup first. It reads a provider table of
the same shape the app uses; the next cell defines the two entries these
examples need and runs preflight over five configurations: the offline stub,
a provider with and without a key, a local provider that needs no key, and an
unknown provider.

In [6]:
PLATFORMS = {
    "openai": ("OPENAI_API_KEY", "gpt-4.1-mini-2025-04-14"),
    "ollama": (None, "llama3.1:8b"),
}

cases = [
    ("offline stub", preflight(PLATFORMS, None, None, stub=True), True),
    ("openai with no key set", preflight(PLATFORMS, "openai", None, env={}), False),
    ("openai with a key set",
     preflight(PLATFORMS, "openai", None, env={"OPENAI_API_KEY": "sk-test"}), True),
    ("ollama needs no key", preflight(PLATFORMS, "ollama", None, env={}), True),
    ("unknown provider, no model",
     preflight(PLATFORMS, "nosuchprovider", None, env={}), False),
]
for label, result, expected_ok in cases:
    verdict = "ok" if result.ok else "caught before the run"
    detail = "" if result.ok else "; ".join(
        f"{c.name}: {c.detail}" for c in result.checks if not c.ok)
    print(f"{label:28s} -> {verdict}" + (f" ({detail})" if detail else ""))

offline stub                 -> ok
openai with no key set       -> caught before the run (API key: OPENAI_API_KEY is not set. Put it in .env, or tick 'offline stub' to try the UI without a key.)
openai with a key set        -> ok
ollama needs no key          -> ok
unknown provider, no model   -> caught before the run (Provider: 'nosuchprovider' is not in the provider table; it will be used as a custom platform with the model name given.; Model name: No model name given and this provider has no default. Enter a model name.; API key: NOSUCHPROVIDER_API_KEY is not set. Put it in .env, or tick 'offline stub' to try the UI without a key.)


When a call does fail during a run, the error is classified and the user gets
guidance for that category. Temporary failures, such as rate limits, timeouts,
and connection problems, are retried with a growing delay between attempts.
Permanent failures, such as a rejected key, fail at once instead of wasting
retries.

In [7]:
examples = ["429 Too Many Requests", "401 invalid api key",
            "model not found", "request timed out", "connection reset by peer"]
for text in examples:
    category = classify_error(RuntimeError(text))
    print(f"{text:28s} -> {category:16s} {guidance_for(category)}")

429 Too Many Requests        -> rate_limit       The model provider's free-tier limit was reached. Wait a minute and try again, or switch provider under Advanced (free limits: OpenRouter 50 requests/day until $10 of credit is bought, then 1,000/day; Groq about 1,000/day).
401 invalid api key          -> auth             The API key was rejected. Check the key for the chosen provider.
model not found              -> model_not_found  That model name is not available on this provider; clear the Model box to use the provider default.
request timed out            -> timeout          The provider did not answer in time. Try again.
connection reset by peer     -> connection       The provider could not be reached. Check the connection and try again.


In [8]:
def flaky(failures, message):
    state = {"calls": 0}

    def fn():
        state["calls"] += 1
        if state["calls"] <= failures:
            raise RuntimeError(message)
        return "recovered"

    return fn, state


# The sleep is replaced here so the notebook does not wait; the app waits
# 0.5 s, then 1.0 s between attempts.
fn, state = flaky(2, "429 Too Many Requests")
out = call_with_retry(fn, sleep=lambda s: None,
                      on_retry=lambda a, c, e: print(
                          f"attempt {a} failed ({c}); retrying"))
print(f"Rate limited twice, then: {out} after {state['calls']} attempts")

auth_fn, auth_state = flaky(5, "401 invalid api key")
try:
    call_with_retry(auth_fn, sleep=lambda s: None)
except RuntimeError:
    print(f"Rejected key: gave up after {auth_state['calls']} attempt, no retries")

attempt 1 failed (rate_limit); retrying
attempt 2 failed (rate_limit); retrying
Rate limited twice, then: recovered after 3 attempts
Rejected key: gave up after 1 attempt, no retries


## What this demonstrates

Each Phase 2 enhancement, running in the same module code the app imports:

- Sessions survive a restart. The reopened session above restored every saved
  field, its guidance note, and its full transcript exactly, and continued
  from the turn it had reached. Phase 1 restored nothing.
- Progress problems are visible while a run is still going. The monitor
  flagged the repeated haiku as repetition at turn 2 and as a stall at turn 4,
  and flagged the Montreal conversation as drift when it left the task.
- Setup mistakes are caught before they cost a run. Preflight passed the
  valid configurations and caught the invalid ones before any model call, and
  retry recovered from simulated rate limits, timeouts, and connection errors
  while failing fast on a rejected key.

Two honest boundaries. First, this notebook inlines the three modules and
demonstrates them directly; the Playground app that wraps them, with its live
sessions, turn limit, pause and resume, and agent mode, needs the CAMEL
framework and Gradio, and is exercised in the repository by
`demo/test_phase2.py` (19 checks) and `demo/benchmark_phase2.py`, the source
of the report's evidence table. Second, the limits stated in the report apply
here as well: the monitor judges word overlap, not meaning, so its flags point
at possible problems rather than proving them, and the hosted deployment
stores sessions on an ephemeral filesystem, so they can disappear when the
service restarts.